# CLIP4CAD-GFA v2 Training

**Key Innovation: Joint Self-Grounding Training with Embedding Distillation**

Self-grounding learns via:
1. **Contrastive loss** (aligns z_self with text)
2. **Embedding distillation** (forces z_self ≈ z_guided) - KEY FIX for collapse!
3. **Grounding distillation** (aligns attention patterns)

## Training Stages
- **Stage 1 (Epochs 1-15)**: Heavy distillation (λ_embed=0.5) to establish alignment
- **Stage 2 (Epochs 16-35)**: Balanced training with hard negatives

## Loss Weights (REBALANCED)
| Stage | λ_self | λ_distill | λ_embed_distill | λ_detail |
|-------|--------|-----------|-----------------|----------|
| 1 | 0.1 | 0.3 | **0.5** | 0.0 |
| 2 | 0.3 | 0.2 | **0.3** | 0.3 |

## Success Criteria
- Stage 1: Self cosine ≥ 0.8 (should NOT collapse!)
- Stage 2: Self cosine ≥ 0.9, Text→BRep R@1 ≥ 55%

In [5]:
# Cell 1: Imports and Setup
import sys
sys.path.insert(0, '..')

import os
import gc
import math
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torch.cuda.amp import GradScaler, autocast
from torch.optim import AdamW
from torch.optim.lr_scheduler import LambdaLR
from tqdm.auto import tqdm
import numpy as np
from pathlib import Path

# Set device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name()}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

Device: cuda
GPU: NVIDIA GeForce RTX 4090
Memory: 25.8 GB


In [6]:
# Cell 2: Data Paths
# Adjust these paths to match your setup

DATA_ROOT = Path("d:/Defect_Det/MMCAD/data")
PC_FILE = Path("c:/Users/User/Desktop/pc_embeddings_full.h5")
BREP_FILE = Path("c:/Users/User/Desktop/brep_features.h5")
OUTPUT_DIR = Path("../outputs/gfa_v2")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Data root: {DATA_ROOT}")
print(f"PC file: {PC_FILE} (exists: {PC_FILE.exists()})")
print(f"BRep file: {BREP_FILE} (exists: {BREP_FILE.exists()})")
print(f"Output: {OUTPUT_DIR}")

Data root: d:\Defect_Det\MMCAD\data
PC file: c:\Users\User\Desktop\pc_embeddings_full.h5 (exists: True)
BRep file: c:\Users\User\Desktop\brep_features.h5 (exists: True)
Output: ..\outputs\gfa_v2


In [7]:
# Cell 3: Load Data using GFAMappedDataset (same as train_gfa.ipynb)

from clip4cad.data.gfa_dataset import GFAMappedDataset

# Single text file (not split dir)
TEXT_FILE = Path("c:/Users/User/Desktop/text_embeddings.h5")

print("Loading datasets...")
print("=" * 60)

# Train dataset - LOAD TO RAM for fast training
print("\n[1/2] Loading TRAIN dataset to RAM...")
train_dataset = GFAMappedDataset(
    data_root=str(DATA_ROOT),
    split="train",
    pc_file=str(PC_FILE),
    text_file=str(TEXT_FILE),
    brep_file=str(BREP_FILE),
    num_rotations=1,
    load_to_memory=True,
    use_live_text=False,
)
print(f"Train: {len(train_dataset):,} samples in RAM")

# Val dataset - ON DISK (saves RAM)
print("\n[2/2] Loading VAL dataset (on disk)...")
val_dataset = GFAMappedDataset(
    data_root=str(DATA_ROOT),
    split="val",
    pc_file=str(PC_FILE),
    text_file=str(TEXT_FILE),
    brep_file=str(BREP_FILE),
    num_rotations=1,
    load_to_memory=False,
    use_live_text=False,
)
print(f"Val: {len(val_dataset):,} samples on disk")

print("\n" + "=" * 60)
print("DATASETS READY!")
print("=" * 60)

Loading datasets...

[1/2] Loading TRAIN dataset to RAM...
  Loading train data to memory (B-Rep + PC + Text)...
    Loading B-Rep (3GB)...
    Loading PC (50GB)...
    Loading Text from: c:\Users\User\Desktop\text_splits\train_text_embeddings.h5
    ✓ Text loaded: 174.6GB in 366.3s
    ⚠️  Pre-split has 111000 samples, dataset expected 133105
    Using first 111000 samples to match pre-split file
  ✓ Loaded 111000 samples: 203.6GB in RAM (B-Rep + PC + Text)
GFAMappedDataset: train with 111000 samples (in memory)
Train: 111,000 samples in RAM

[2/2] Loading VAL dataset (on disk)...
GFAMappedDataset: val with 16638 samples
Val: 16,638 samples on disk

DATASETS READY!


In [8]:
# Cell 4: Verify Dataset

# Quick sanity check - make sure data loads correctly
sample = train_dataset[0]
print(f"Sample keys: {list(sample.keys())}")
print(f"  brep_face_features: {sample['brep_face_features'].shape}")
print(f"  brep_edge_features: {sample['brep_edge_features'].shape}")
print(f"  brep_face_mask: {sample['brep_face_mask'].shape}")
print(f"  pc_features: {sample['pc_features'].shape}")
print(f"  desc_embedding: {sample['desc_embedding'].shape}")

Sample keys: ['sample_id', 'idx', 'rot_idx', 'brep_face_features', 'brep_edge_features', 'brep_face_mask', 'brep_edge_mask', 'pc_features', 'desc_embedding', 'desc_mask', 'has_brep', 'use_cached_brep_features', 'has_pointcloud', 'use_cached_pc_features', 'has_text', 'use_cached_embeddings']
  brep_face_features: torch.Size([192, 48])
  brep_edge_features: torch.Size([512, 12])
  brep_face_mask: torch.Size([192])
  pc_features: torch.Size([48, 1024])
  desc_embedding: torch.Size([256, 3072])


In [9]:
# Cell 5: Dataset Summary

print(f"Train dataset: {len(train_dataset)} samples")
print(f"Val dataset: {len(val_dataset)} samples")

Train dataset: 111000 samples
Val dataset: 16638 samples


In [10]:
# Cell 6: Create Model

from clip4cad.models import CLIP4CAD_GFA_v2, GFAv2Config
from clip4cad.losses import GFAv2Loss, compute_self_grounding_quality

# Configuration
config = GFAv2Config(
    d_face=48,          # FSQ face features
    d_edge=12,          # FSQ edge features
    d_pc=1024,          # ShapeLLM features (actual dim from HDF5)
    d_text=3072,        # Phi-4-mini features
    d_unified=256,
    d_proj=128,
    d_ground=128,
    num_slots=12,
    num_detail_queries=8,
    num_heads=8,
    num_parser_layers=2,
    num_self_ground_layers=2,
    dropout=0.1,
)

model = CLIP4CAD_GFA_v2(config).to(device)
print(f"Model parameters: {model.count_parameters():,}")
print(f"Trainable parameters: {model.count_parameters(trainable_only=True):,}")

Model parameters: 8,117,640
Trainable parameters: 8,117,640


In [ ]:
# Cell 7: Training Configuration

from clip4cad.data.gfa_dataset import gfa_collate_fn

# Hyperparameters
BATCH_SIZE = 512  # Reduce if running out of memory
STAGE1_EPOCHS = 15
STAGE2_EPOCHS = 20
STAGE1_LR = 3e-5
STAGE2_LR = 1e-5
WARMUP_EPOCHS = 3  # Longer warmup for stability
MAX_GRAD_NORM = 1.0

# Loss weights - REBALANCED to fix self-grounding collapse
# Stage 1: Heavy distillation to establish alignment
STAGE1_LAMBDA_SELF = 0.1              # REDUCED (was 0.3) - let distillation dominate early
STAGE1_LAMBDA_DISTILL = 0.3           # INCREASED (was 0.1) - stronger grounding alignment
STAGE1_LAMBDA_EMBED_DISTILL = 0.5     # NEW! KEY FIX: force z_self ≈ z_guided
STAGE1_LAMBDA_DETAIL = 0.0

# Stage 2: More balanced
STAGE2_LAMBDA_SELF = 0.3
STAGE2_LAMBDA_DISTILL = 0.2
STAGE2_LAMBDA_EMBED_DISTILL = 0.3     # NEW!
STAGE2_LAMBDA_DETAIL = 0.3

# Create data loaders
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,  # Set to 0 for notebook compatibility
    pin_memory=True,
    drop_last=True,
    collate_fn=gfa_collate_fn,
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=True,
    collate_fn=gfa_collate_fn,
)

print(f"Batch size: {BATCH_SIZE}")
print(f"Train batches: {len(train_loader)}")
print(f"Val batches: {len(val_loader)}")
print(f"Total epochs: {STAGE1_EPOCHS + STAGE2_EPOCHS}")
print(f"\nStage 1 weights: self={STAGE1_LAMBDA_SELF}, distill={STAGE1_LAMBDA_DISTILL}, embed_distill={STAGE1_LAMBDA_EMBED_DISTILL}")
print(f"Stage 2 weights: self={STAGE2_LAMBDA_SELF}, distill={STAGE2_LAMBDA_DISTILL}, embed_distill={STAGE2_LAMBDA_EMBED_DISTILL}")

Batch size: 512
Train batches: 216
Val batches: 33
Total epochs: 35

Stage 1 weights: self=0.1, distill=0.3, embed_distill=0.5
Stage 2 weights: self=0.3, distill=0.2, embed_distill=0.3


In [14]:
# Cell 8: Initialize Optimizer, Loss, Scheduler, and Hard Negative Miner
import importlib
import clip4cad.losses.gfa_v2_losses as gfa_v2_losses_module
importlib.reload(gfa_v2_losses_module)
from clip4cad.losses.gfa_v2_losses import GFAv2Loss
from clip4cad.training.hard_negative_mining import HardNegativeMiner

optimizer = AdamW(
    model.parameters(),
    lr=STAGE1_LR,
    weight_decay=0.01,
    betas=(0.9, 0.999),
)

criterion = GFAv2Loss(
    lambda_self=STAGE1_LAMBDA_SELF,
    lambda_distill=STAGE1_LAMBDA_DISTILL,
    lambda_embed_distill=STAGE1_LAMBDA_EMBED_DISTILL,  # NEW! Key fix for collapse
    lambda_detail=STAGE1_LAMBDA_DETAIL,
)

scaler = GradScaler()

# Learning rate scheduler with warmup
total_epochs = STAGE1_EPOCHS + STAGE2_EPOCHS
warmup_steps = WARMUP_EPOCHS * len(train_loader)
total_steps = total_epochs * len(train_loader)

def lr_lambda(step):
    if step < warmup_steps:
        return step / max(warmup_steps, 1)
    progress = (step - warmup_steps) / max(total_steps - warmup_steps, 1)
    return max(1e-6 / STAGE1_LR, 0.5 * (1 + math.cos(math.pi * progress)))

scheduler = LambdaLR(optimizer, lr_lambda)

# Hard negative miner (used in Stage 2)
hard_neg_miner = HardNegativeMiner(
    model=model,
    train_dataloader=train_loader,
    cache_dir=str(OUTPUT_DIR / "hard_negatives"),
    k=20,
    text_sim_threshold=0.8,
    min_negatives=1,
    max_negatives=10,
    use_faiss=True,
    device=str(device),
)
hard_negatives = None
MINE_EVERY_N_EPOCHS = 5

print("Optimizer, loss, scheduler, and hard negative miner initialized.")
print(f"Embed distillation weight: {STAGE1_LAMBDA_EMBED_DISTILL} (KEY FIX for collapse)")

Optimizer, loss, scheduler, and hard negative miner initialized.
Embed distillation weight: 0.5 (KEY FIX for collapse)


C:\Users\User\AppData\Local\Temp\ipykernel_21672\2817317415.py:22: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler()


In [13]:
# Cell 9: Training Loop

# Training state
global_step = 0
best_val_loss = float('inf')
best_self_cosine = 0.0
current_stage = 1

# Training history
history = {
    'train_loss': [],
    'val_loss': [],
    'self_cosine_brep': [],
    'self_cosine_pc': [],
    'embed_distill': [],
}

print("Starting training...")
print("="*70)

for epoch in range(1, total_epochs + 1):
    # Stage transition
    if epoch == STAGE1_EPOCHS + 1:
        print("\n" + "="*70)
        print("TRANSITIONING TO STAGE 2")
        print("="*70)
        current_stage = 2
        
        # Update loss weights (including embed_distill)
        criterion.update_weights(
            lambda_self=STAGE2_LAMBDA_SELF,
            lambda_distill=STAGE2_LAMBDA_DISTILL,
            lambda_embed_distill=STAGE2_LAMBDA_EMBED_DISTILL,  # NEW!
            lambda_detail=STAGE2_LAMBDA_DETAIL,
        )
        print(f"Updated loss weights: self={STAGE2_LAMBDA_SELF}, distill={STAGE2_LAMBDA_DISTILL}, embed_distill={STAGE2_LAMBDA_EMBED_DISTILL}, detail={STAGE2_LAMBDA_DETAIL}")
        
        # Reduce learning rate
        for param_group in optimizer.param_groups:
            param_group['lr'] = STAGE2_LR
        print(f"Reduced LR to {STAGE2_LR}")
        
        # Save Stage 1 checkpoint
        torch.save({
            'epoch': epoch - 1,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'best_self_cosine': best_self_cosine,
        }, OUTPUT_DIR / 'checkpoint_stage1_final.pt')
        print(f"Saved Stage 1 checkpoint")
        
        # Mine hard negatives at start of Stage 2
        print("\nMining hard negatives for Stage 2...")
        hard_negatives = hard_neg_miner.mine(epoch=epoch)
        print(f"Mined hard negatives for {len(hard_negatives)} samples")
    
    # Re-mine hard negatives every N epochs in Stage 2
    if current_stage == 2 and epoch > STAGE1_EPOCHS + 1:
        if (epoch - STAGE1_EPOCHS - 1) % MINE_EVERY_N_EPOCHS == 0:
            print(f"\nRe-mining hard negatives (epoch {epoch})...")
            hard_negatives = hard_neg_miner.mine(epoch=epoch)
            print(f"Re-mined hard negatives for {len(hard_negatives)} samples")
    
    # Train epoch
    model.train()
    epoch_loss = 0.0
    epoch_self_cos_brep = []
    epoch_self_cos_pc = []
    epoch_embed_distill = []
    
    pbar = tqdm(train_loader, desc=f"Epoch {epoch} (Stage {current_stage})")
    for batch_idx, batch in enumerate(pbar):
        # Get hard negatives for this batch (if in Stage 2)
        batch_hard_negs = None
        if current_stage == 2 and hard_negatives is not None:
            # Map batch indices to hard negatives
            batch_size = batch['brep_face_features'].shape[0]
            start_idx = batch_idx * BATCH_SIZE
            batch_hard_negs = []
            for i in range(batch_size):
                sample_idx = start_idx + i
                if sample_idx in hard_negatives:
                    batch_hard_negs.append(hard_negatives[sample_idx])
                else:
                    batch_hard_negs.append(None)
        
        with autocast():
            outputs = model(batch)
            loss, loss_dict = criterion(outputs, hard_negatives=batch_hard_negs, stage=current_stage)
        
        optimizer.zero_grad()
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), MAX_GRAD_NORM)
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()
        
        global_step += 1
        epoch_loss += loss_dict['total']
        epoch_embed_distill.append(loss_dict.get('embed_distill', 0))
        
        # Compute self-grounding quality
        if outputs.get('z_brep') is not None and outputs.get('z_brep_self') is not None:
            cos_brep = compute_self_grounding_quality(
                outputs['z_brep'].detach(),
                outputs['z_brep_self'].detach()
            )
            epoch_self_cos_brep.append(cos_brep)
        
        if outputs.get('z_pc') is not None and outputs.get('z_pc_self') is not None:
            cos_pc = compute_self_grounding_quality(
                outputs['z_pc'].detach(),
                outputs['z_pc_self'].detach()
            )
            epoch_self_cos_pc.append(cos_pc)
        
        # Update progress bar (with new embed_d metric)
        postfix = {
            'loss': f"{loss_dict['total']:.3f}",
            'guided': f"{loss_dict['guided']:.3f}",
            'self': f"{loss_dict['self']:.3f}",
            'emb_d': f"{loss_dict.get('embed_distill', 0):.3f}",  # NEW: track embed distill
            'cos': f"{epoch_self_cos_brep[-1]:.3f}" if epoch_self_cos_brep else "N/A",
        }
        if current_stage == 2:
            postfix['detail'] = f"{loss_dict.get('detail', 0):.3f}"
        pbar.set_postfix(postfix)
    
    # Epoch summary
    avg_loss = epoch_loss / len(train_loader)
    avg_cos_brep = sum(epoch_self_cos_brep) / len(epoch_self_cos_brep) if epoch_self_cos_brep else 0
    avg_cos_pc = sum(epoch_self_cos_pc) / len(epoch_self_cos_pc) if epoch_self_cos_pc else 0
    avg_embed_distill = sum(epoch_embed_distill) / len(epoch_embed_distill) if epoch_embed_distill else 0
    
    history['train_loss'].append(avg_loss)
    history['self_cosine_brep'].append(avg_cos_brep)
    history['self_cosine_pc'].append(avg_cos_pc)
    history['embed_distill'].append(avg_embed_distill)
    
    if avg_cos_brep > best_self_cosine:
        best_self_cosine = avg_cos_brep
    
    print(f"\nEpoch {epoch}: Loss={avg_loss:.4f}, Self-cos BRep={avg_cos_brep:.4f}, Self-cos PC={avg_cos_pc:.4f}, Embed-distill={avg_embed_distill:.4f}")
    print(f"Best self-cosine: {best_self_cosine:.4f}")
    
    # Validation every 5 epochs
    if epoch % 5 == 0:
        model.eval()
        val_loss = 0.0
        val_cos_brep = []
        
        with torch.no_grad():
            for batch in tqdm(val_loader, desc="Validation"):
                with autocast():
                    outputs = model(batch)
                    loss, loss_dict = criterion(outputs, stage=current_stage)
                val_loss += loss_dict['total']
                
                if outputs.get('z_brep') is not None and outputs.get('z_brep_self') is not None:
                    cos_brep = compute_self_grounding_quality(
                        outputs['z_brep'],
                        outputs['z_brep_self']
                    )
                    val_cos_brep.append(cos_brep)
        
        avg_val_loss = val_loss / len(val_loader)
        avg_val_cos = sum(val_cos_brep) / len(val_cos_brep) if val_cos_brep else 0
        
        history['val_loss'].append(avg_val_loss)
        print(f"Validation: Loss={avg_val_loss:.4f}, Self-cos={avg_val_cos:.4f}")
        
        # Save best model
        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            torch.save({
                'epoch': epoch,
                'model_state_dict': model.state_dict(),
                'best_val_loss': best_val_loss,
                'best_self_cosine': best_self_cosine,
            }, OUTPUT_DIR / 'checkpoint_best.pt')
            print("Saved best model!")
    
    # Save checkpoint every 5 epochs
    if epoch % 5 == 0:
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'scheduler_state_dict': scheduler.state_dict(),
            'best_self_cosine': best_self_cosine,
        }, OUTPUT_DIR / f'checkpoint_epoch{epoch}.pt')
    
    # Clear cache
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        gc.collect()

print("\n" + "="*70)
print("Training Complete!")
print(f"Best self-grounding cosine: {best_self_cosine:.4f}")
print(f"Best validation loss: {best_val_loss:.4f}")
print("="*70)

Starting training...


Epoch 1 (Stage 1):   0%|          | 0/216 [00:00<?, ?it/s]

C:\Users\User\AppData\Local\Temp\ipykernel_21672\2588864086.py:87: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
c:\Users\User\anaconda3\envs\clip4cad\lib\site-packages\torch\optim\lr_scheduler.py:224: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  warnings.warn(



Epoch 1: Loss=7.5006, Self-cos BRep=0.8766, Self-cos PC=0.9031, Embed-distill=0.1101
Best self-cosine: 0.8766


Epoch 2 (Stage 1):   0%|          | 0/216 [00:00<?, ?it/s]

...

Mining hard negatives for 110592 samples...
  Computing geometric kNN...
  Filtering by text similarity...


  Filtering: 100%|██████████| 110592/110592 [00:04<00:00, 22543.98it/s]


  Found hard negatives for 92081 samples
  Average negatives per sample: 4.7
Saved hard negatives to ..\outputs\gfa_v2\hard_negatives\hard_negatives_epoch16.json

Mined hard negatives for 92081 samples


Epoch 16 (Stage 2):   0%|          | 0/216 [00:00<?, ?it/s]

IndexError: index 64935 is out of bounds for dimension 1 with size 512

In [17]:
# Resume from epoch 16
START_EPOCH = 16
current_stage = 2
criterion.lambda_detail = 0.3

print(f"Resuming training from epoch {START_EPOCH}...")
print("="*70)

for epoch in range(START_EPOCH, total_epochs + 1):
    # Re-mine hard negatives every N epochs in Stage 2
    if epoch > STAGE1_EPOCHS + 1 and (epoch - STAGE1_EPOCHS - 1) % MINE_EVERY_N_EPOCHS == 0:
        print(f"\nRe-mining hard negatives (epoch {epoch})...")
        hard_negatives = hard_neg_miner.mine(epoch=epoch)
        print(f"Re-mined hard negatives for {len(hard_negatives)} samples")

    # Train epoch
    model.train()
    epoch_loss = 0.0
    epoch_self_cos_brep = []
    epoch_self_cos_pc = []
    epoch_embed_distill = []

    pbar = tqdm(train_loader, desc=f"Epoch {epoch} (Stage {current_stage})")
    for batch_idx, batch in enumerate(pbar):
        # Get hard negatives for this batch
        batch_hard_negs = None
        if hard_negatives is not None:
            batch_size = batch['brep_face_features'].shape[0]
            start_idx = batch_idx * BATCH_SIZE
            batch_hard_negs = []
            for i in range(batch_size):
                sample_idx = start_idx + i
                if sample_idx in hard_negatives:
                    batch_hard_negs.append(hard_negatives[sample_idx])
                else:
                    batch_hard_negs.append(None)

        with autocast():
            outputs = model(batch)
            loss, loss_dict = criterion(outputs, hard_negatives=batch_hard_negs, stage=current_stage)

        optimizer.zero_grad()
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), MAX_GRAD_NORM)
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()

        global_step += 1
        epoch_loss += loss_dict['total']
        epoch_embed_distill.append(loss_dict.get('embed_distill', 0))

        if outputs.get('z_brep') is not None and outputs.get('z_brep_self') is not None:
            cos_brep = compute_self_grounding_quality(outputs['z_brep'].detach(), outputs['z_brep_self'].detach())
            epoch_self_cos_brep.append(cos_brep)

        if outputs.get('z_pc') is not None and outputs.get('z_pc_self') is not None:
            cos_pc = compute_self_grounding_quality(outputs['z_pc'].detach(), outputs['z_pc_self'].detach())
            epoch_self_cos_pc.append(cos_pc)

        postfix = {
            'loss': f"{loss_dict['total']:.3f}",
            'guided': f"{loss_dict['guided']:.3f}",
            'self': f"{loss_dict['self']:.3f}",
            'emb_d': f"{loss_dict.get('embed_distill', 0):.3f}",
            'detail': f"{loss_dict.get('detail', 0):.3f}",
            'cos': f"{epoch_self_cos_brep[-1]:.3f}" if epoch_self_cos_brep else "N/A",
        }
        pbar.set_postfix(postfix)

    # Epoch summary
    avg_loss = epoch_loss / len(train_loader)
    avg_cos_brep = sum(epoch_self_cos_brep) / len(epoch_self_cos_brep) if epoch_self_cos_brep else 0
    avg_cos_pc = sum(epoch_self_cos_pc) / len(epoch_self_cos_pc) if epoch_self_cos_pc else 0
    avg_embed_distill = sum(epoch_embed_distill) / len(epoch_embed_distill) if epoch_embed_distill else 0

    history['train_loss'].append(avg_loss)
    history['self_cosine_brep'].append(avg_cos_brep)
    history['self_cosine_pc'].append(avg_cos_pc)
    history['embed_distill'].append(avg_embed_distill)

    if avg_cos_brep > best_self_cosine:
        best_self_cosine = avg_cos_brep

    print(f"\nEpoch {epoch}: Loss={avg_loss:.4f}, Self-cos BRep={avg_cos_brep:.4f}, Self-cos PC={avg_cos_pc:.4f}, Embed-distill={avg_embed_distill:.4f}")
    print(f"Best self-cosine: {best_self_cosine:.4f}")

    # Validation every 5 epochs
    if epoch % 5 == 0:
        model.eval()
        val_loss = 0.0
        val_cos_brep = []

        with torch.no_grad():
            for batch in tqdm(val_loader, desc="Validation"):
                with autocast():
                    outputs = model(batch)
                    loss, loss_dict = criterion(outputs, stage=current_stage)
                val_loss += loss_dict['total']

                if outputs.get('z_brep') is not None and outputs.get('z_brep_self') is not None:
                    val_cos_brep.append(compute_self_grounding_quality(outputs['z_brep'], outputs['z_brep_self']))

        avg_val_loss = val_loss / len(val_loader)
        avg_val_cos = sum(val_cos_brep) / len(val_cos_brep) if val_cos_brep else 0
        history['val_loss'].append(avg_val_loss)
        print(f"Validation: Loss={avg_val_loss:.4f}, Self-cos={avg_val_cos:.4f}")

        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            torch.save({'epoch': epoch, 'model_state_dict': model.state_dict(), 'best_val_loss': best_val_loss, 'best_self_cosine': best_self_cosine}, OUTPUT_DIR / 'checkpoint_best.pt')
            print("Saved best model!")

    if epoch % 5 == 0:
        torch.save({'epoch': epoch, 'model_state_dict': model.state_dict(), 'optimizer_state_dict': optimizer.state_dict(), 'scheduler_state_dict': scheduler.state_dict(), 'best_self_cosine': best_self_cosine}, OUTPUT_DIR /
f'checkpoint_epoch{epoch}.pt')

    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        gc.collect()

print("\n" + "="*70)
print("Training Complete!")
print(f"Best self-grounding cosine: {best_self_cosine:.4f}")
print(f"Best validation loss: {best_val_loss:.4f}")
print("="*70)

Resuming training from epoch 16...


Epoch 16 (Stage 2):   0%|          | 0/216 [00:00<?, ?it/s]

C:\Users\User\AppData\Local\Temp\ipykernel_21672\2369034567.py:38: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():



Epoch 16: Loss=5.2034, Self-cos BRep=0.2005, Self-cos PC=0.3603, Embed-distill=0.7196
Best self-cosine: 0.8766


Epoch 17 (Stage 2):   0%|          | 0/216 [00:00<?, ?it/s]

...


Epoch 34: Loss=4.0132, Self-cos BRep=0.1973, Self-cos PC=0.3610, Embed-distill=0.7208
Best self-cosine: 0.8766


Epoch 35 (Stage 2):   0%|          | 0/216 [00:00<?, ?it/s]


Epoch 35: Loss=3.9835, Self-cos BRep=0.1989, Self-cos PC=0.3624, Embed-distill=0.7194
Best self-cosine: 0.8766


Validation:   0%|          | 0/33 [00:00<?, ?it/s]

Validation: Loss=3.6412, Self-cos=0.1427

Training Complete!
Best self-grounding cosine: 0.8766
Best validation loss: 3.6272


In [16]:
# Debug: Check what's being output                                                                                                                                                                                                           model.eval()
with torch.no_grad():
    sample_batch = next(iter(train_loader))
    with autocast():
        outputs = model(sample_batch)                                                                                                                                                                                                      
    print("Output keys:", list(outputs.keys()))
    print(f"z_brep_detail: {outputs.get('z_brep_detail') is not None}")
    print(f"z_pc_detail: {outputs.get('z_pc_detail') is not None}")
    print(f"z_text: {outputs.get('z_text') is not None}")
    print(f"lambda_detail: {criterion.lambda_detail}")

    if outputs.get('z_brep_detail') is not None:
        print(f"z_brep_detail shape: {outputs['z_brep_detail'].shape}")

model.train()

C:\Users\User\AppData\Local\Temp\ipykernel_21672\3474307423.py:4: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():


Output keys: ['z_brep', 'z_pc', 'z_text', 'z_brep_self', 'z_pc_self', 'z_brep_detail', 'z_pc_detail', 'G_brep_guided', 'G_pc_guided', 'G_brep_self', 'G_pc_self', 'confidence', 'confidence_brep_self', 'confidence_pc_self', 'tau']
z_brep_detail: True
z_pc_detail: True
z_text: True
lambda_detail: 0.0
z_brep_detail shape: torch.Size([512, 128])


CLIP4CAD_GFA_v2(
  (brep_proj): BRepProjection(
    (proj_face): Sequential(
      (0): Linear(in_features=48, out_features=256, bias=True)
      (1): GELU(approximate='none')
      (2): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
      (3): Dropout(p=0.1, inplace=False)
      (4): Linear(in_features=256, out_features=256, bias=True)
      (5): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
      (6): Dropout(p=0.1, inplace=False)
    )
    (proj_edge): Sequential(
      (0): Linear(in_features=12, out_features=128, bias=True)
      (1): GELU(approximate='none')
      (2): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
      (3): Dropout(p=0.1, inplace=False)
      (4): Linear(in_features=128, out_features=256, bias=True)
      (5): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
      (6): Dropout(p=0.1, inplace=False)
    )
  )
  (pc_proj): PCProjection(
    (proj_local): Sequential(
      (0): Linear(in_features=1024, out_features=256, bias=True)
      

In [18]:
# Cell 10: Save Final Model

torch.save({
    'model_state_dict': model.state_dict(),
    'config': config.__dict__,
    'best_self_cosine': best_self_cosine,
    'history': history,
}, OUTPUT_DIR / 'clip4cad_gfa_v2_final.pt')

print(f"Final model saved to {OUTPUT_DIR / 'clip4cad_gfa_v2_final.pt'}")

Final model saved to ..\outputs\gfa_v2\clip4cad_gfa_v2_final.pt


In [ ]:
# Cell 11: Plot Training History

import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Loss plot
axes[0].plot(history['train_loss'], label='Train Loss')
if history['val_loss']:
    val_epochs = list(range(5, len(history['train_loss']) + 1, 5))[:len(history['val_loss'])]
    axes[0].plot(val_epochs, history['val_loss'], 'o-', label='Val Loss')
axes[0].axvline(x=STAGE1_EPOCHS, color='r', linestyle='--', label='Stage Transition')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('Training Loss')
axes[0].legend()
axes[0].grid(True)

# Self-grounding quality plot
axes[1].plot(history['self_cosine_brep'], label='BRep Self-Cosine')
axes[1].plot(history['self_cosine_pc'], label='PC Self-Cosine')
axes[1].axvline(x=STAGE1_EPOCHS, color='r', linestyle='--', label='Stage Transition')
axes[1].axhline(y=0.9, color='g', linestyle=':', label='Target (0.9)')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Cosine Similarity')
axes[1].set_title('Self-Grounding Quality')
axes[1].legend()
axes[1].grid(True)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'training_history.png', dpi=150)
plt.show()

print(f"Plot saved to {OUTPUT_DIR / 'training_history.png'}")